**Demo plotting results from CVP Results**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os
import cv2
import ast

**Example Reading SQL Database**

In [ ]:
query = '''
SELECT * 
FROM cvp_results
''' 
conn_str = 'sqlite:///../data/cvp_database.db'
df = pd.read_sql(query, conn_str)
df

**Example Reading CSV File**

In [ ]:
df = pd.read_csv(r"..\data\cvp_model_results.csv")
df.head(5)

**Plot Results with Bounding Boxes from Images**

In [ ]:
def add_rect(coords, color):
    if len(coords) < 4:
        return None
    ymax, xmax = float(coords[3]),float(coords[2])
    ymin, xmin = float(coords[1]),float(coords[0])

    width = xmax-xmin
    height = ymax-ymin

    # create rectangle obejct
    rect = patches.Rectangle(
        (xmin, ymin), width, height, 
        linewidth=1, edgecolor=color, facecolor='none', alpha = 0.7)

    return rect

for idx, row in df.iterrows():
    if idx > 5:
        break
    fp = os.path.join(r"..\data\license_plate_detection\train\images", f"{row["file_name"]}.jpg")
    img  = cv2.cvtColor(cv2.imread(fp), cv2.COLOR_BGR2RGB)
    fig, axes = plt.subplots()
    axes.imshow(img)

    coords = ast.literal_eval(row["yolo_xy_coords"])
    rect = add_rect(coords, "red")

    try:
        coords_ocr = ast.literal_eval(row["ocr_bbox"])
    except:
        coords_ocr =[]
    rect1 = add_rect(coords_ocr, "blue")
    # add rectangle to plot
    axes.add_patch(rect)
    if rect1 is not None:
        axes.add_patch(rect1)

    axes.set_xticks([])
    axes.set_yticks([])
    axes.set_title(f"{row['file_name']}")


    plt.show()
    plt.close()
    